# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 01 · Repository Ingestion

This notebook demonstrates the **repository ingestion stage** of the RAG pipeline.

Given a public GitHub repository URL, the `GitHubRepositoryLoader` retrieves the repository files and converts them into a collection of LangChain `Document` objects.

The implementation of the ingestion stage is located in *src/github_rag/ingestion*:
* `repository.py`: Implements the `GitHubRepositoryLoader` class.
* `filters.py`: Defines the file types and files allowed for ingestion into the RAG pipeline.

#### **Notebook objectives**

* Load a GitHub repository using the `GitHubRepositoryLoader`.
* Explore the structure of the resulting `Document` objects:

  * Raw document content.
  * Metadata associated with each document.

## Imports and Configuration

In [1]:
from collections import Counter
import pandas as pd
from github_rag.ingestion.repository import GitHubRepositoryLoader

You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects, which combines XGBoost and SHAP to build a predictive and explainable system for classifying urban typologies.

In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/urban-typology-xai"

## 1. Repository Ingestion

This section demonstrates the basic behaviour of the `GitHubRepositoryLoader` by loading a public GitHub repository and inspecting the resulting LangChain `Document` objects.

In [3]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")

Number of documents loaded: 28


In [5]:
# Count the number of files for each type
file_types = Counter(
    document.metadata["file_type"]
    for document in documents
)

file_types_df = pd.DataFrame(
    file_types.items(),
    columns=["File type", "Documents"],
).sort_values(
    "Documents",
    ascending=False,
).reset_index(drop=True)

print("Document type distribution:")
file_types_df

Document type distribution:


,File type,Documents
0,.py,13
1,.ipynb,9
2,.txt,2
3,.toml,2
4,.md,1
5,.yml,1


## 2. Exploring LangChain Documents

The ingestion stage returns a list of LangChain `Document` objects.

Each `Document` contains two main components:
- `page_content`: the textual content of the file.
- `metadata`: information about the document's origin and type.

In [7]:
# Select a document to explore
document = documents[0]
print(type(document))

<class 'langchain_core.documents.base.Document'>


In [9]:
# Show the page content
print("Page content:")
print("-"*30)
print(document.page_content[:1000])
print("-"*30)

Page content:
------------------------------
[build-system]
requires = ["setuptools>=61.0.0"]
build-backend = "setuptools.build_meta"

[project]
name = "urban-typology-xai"
version = "0.1.0"
description = "Explainable machine learning for the classification of residential enclosure patterns"
readme = "README.md"
requires-python = ">=3.11"

[tool.setuptools]
packages = ["src"]
------------------------------


In [10]:
# Show the Metadata
print("\nMetadata:")
document.metadata


Metadata:


{'repository': 'davidfernxndez/urban-typology-xai',
 'file_path': 'pyproject.toml',
 'file_type': '.toml',
 'source': 'https://github.com/davidfernxndez/urban-typology-xai/blob/main/pyproject.toml',
 'branch': 'main'}

As we can see, the ingestion stage stores the following metadata:
- **repository**: repository name.
- **file_path**: path of the file within the repository.
- **file_type**: file extension.
- **source**: direct GitHub URL of the file.
- **branch**: checked-out repository branch.

This metadata will later be preserved throughout the RAG pipeline and used to identify the source of retrieved chunks.

We can inspect more documents:

In [11]:
for i, document in enumerate(documents[1:5]):
    print(f"--- Document {i} ---")
    print(f"File: {document.metadata['file_path']}")
    print(f"Type: {document.metadata['file_type']}")
    print(f"Source: {document.metadata['source']}")
    print(f"Content preview:\n{document.page_content[:300]}")
    print()

--- Document 0 ---
File: README.md
Type: .md
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/README.md
Content preview:
# Explainable Artificial Intelligence for the Study of Urban Fragmentation

### A Comparative Analysis of the Performance and Interpretability of Supervised Learning Models for the Classification of Residential Enclosure Patterns

<p align="left">
  <img src="https://img.shields.io/badge/Language-Py

--- Document 1 ---
File: requirements-full.txt
Type: .txt
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/requirements-full.txt
Content preview:
# ==============================================================================
# COMPLETE DEVELOPMENT CONFIGURATION (LOCAL)
#
# This file contains all the project dependencies, covering both the
# experimentation phase and the application of Streamlit.
# ===========================================

--- Document 2 ---
File: requirements.txt
Type: .txt
Source: https://github.com/david